This notebook trains the 4 gene-specific RNA-Protein linear models, estimates the protein count in the expression dataset and creates all plots regarding the linear models for protein expression estimation. This notebooks contains the code to generate the plots from Figure 2A and appendix Figures S1 and S13. Parameters of the linear models are found in Appendix Table S1.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr,pearsonr
from sklearn.linear_model import TheilSenRegressor
from sklearn.metrics import r2_score
import scipy.stats as stats

In [2]:
# Get expression data 
df = pd.read_csv("data/expression/whole_dataset.tsv.gz", sep="\t", compression="gzip")
df = df.loc[df["Gene"].isin(["I10R1_HUMAN","I10R2_HUMAN","STAT1_HUMAN","STAT3_HUMAN"])]
df = df[~df["Cell"].isin(["T4","T4.M","T8","T8.M"])]
cells_add = ["B.","MO.","T4.CM","T8.naive"]
for cell in cells_add:
    df_mean = df.loc[df["Cell"].str.contains(cell)][["Gene","Log2 TPM",'Log10 Prot. count']].groupby("Gene").mean()
    df_mean["Gene"] = df_mean.index
    df_mean = df_mean.reset_index(drop=True)
    df_mean[['Dataset type','RNA dataset num.','Prot. count dataset num.','Cell']] = list(df.loc[df["Cell"].str.contains(cell)].values[1][:3])+[cell+"Mean"]
    df = pd.concat([df, df_mean], ignore_index=True)
df["Cell"] = df["Cell"].replace("T4.CMMean", "T4.Mean")
df["Cell"] = df["Cell"].replace("T8.naiveMean", "T8.Mean")

### Training of linear models for IL10RA, IL10RB, STAT1 and STAT3

In [3]:
# Add data on IL10RA and IL10RB from our experiments
cells = ["ACH-000406","ACH-000509","ACH-000551","ACH-000146"]
df.loc[(df["Cell"].isin(cells)) & (df["Gene"]=="I10R1_HUMAN"),"Log10 Prot. count"] = np.log10([7.64E+02,2.30E+03,6.38E+03,2.59E+03])-1 # 10% of receptors in the surface applied
df.loc[(df["Cell"].isin(cells)) & (df["Gene"]=="I10R1_HUMAN"),"Prot. count dataset num."] = 7
df.loc[(df["Cell"].isin(cells)) & (df["Gene"]=="I10R2_HUMAN"),"Log10 Prot. count"] = np.log10([3.38E+03,4.25E+03,1.10E+04,2.42E+04])-1 # 10% of receptors in the surface applied
df.loc[(df["Cell"].isin(cells)) & (df["Gene"]=="I10R2_HUMAN"),"Prot. count dataset num."] = 7
df.loc[(df["Cell"].isin(["MO.Mean","T4.Mean","T8.Mean","B.Mean","NK"]))&(df["Gene"]=="I10R1_HUMAN"),"Log10 Prot. count"] = np.log10([670,405,750,800,800])
df.loc[(df["Cell"].isin(["MO.Mean","T4.Mean","T8.Mean","B.Mean","NK"])) & (df["Gene"]=="I10R1_HUMAN"),"Prot. count dataset num."] = 8
# Generate the gene-specific linear models
df_lin = pd.DataFrame(columns=["Gene","Intercept","Slope","PCC"])
MAE_dict = {}
gene_list = ["I10R1_HUMAN","I10R2_HUMAN","STAT1_HUMAN","STAT3_HUMAN"]
for gene in gene_list:
    df_gene = df.loc[df["Gene"]==gene].dropna()
    TPM = df_gene["Log2 TPM"]
    Prot = df_gene["Log10 Prot. count"]
    model = TheilSenRegressor(random_state=0).fit(np.array(TPM).reshape(-1, 1), np.array(Prot).reshape(-1, 1))
    df_lin.loc[len(df_lin.index)] = [gene, model.intercept_, model.coef_[0], pearsonr(TPM, Prot).statistic]
    MAE_dict[gene] = np.mean(abs(model.intercept_+TPM.values*model.coef_[0]-Prot.values))
    print(gene + ": MAE= "+str(np.mean(abs(model.intercept_+TPM*model.coef_[0]-Prot))))
    print(gene + ": PCC= "+str(pearsonr(model.intercept_+TPM*model.coef_[0],Prot)))
df_lin.to_csv('data/lin_models/linear_models_IL10.csv')

# Plot the linear models + data used to train it
color_sim = [(68/255, 138/255, 255/255),(0, 150/255, 136/255),(139/255, 195/255, 74/255),(255/255, 193/255, 7/255),(255/255, 152/255, 0),(244/255, 67/255, 54/255),(173/255, 20/255, 87/255)]
fig, ax = plt.subplots(1,1,figsize=(9, 8), dpi=600)
i = 0
for gene in df_lin["Gene"]:
    df_gene = df.loc[df["Gene"] == gene]
    TPM = np.linspace(0, 12, 40)
    Prot = df_lin.loc[df_lin["Gene"]==gene]["Intercept"].values[0] + TPM * df_lin.loc[df_lin["Gene"]==gene]["Slope"].values[0]
    plt.plot(TPM, Prot, color=color_sim[i], linewidth=4)
    plt.scatter(df_gene["Log2 TPM"].values, df_gene["Log10 Prot. count"].values, s=60, color=color_sim[i], alpha = 1, label = gene.split("_HUMAN")[0])
    i += 1
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.xaxis.set_tick_params(width=5, length=10)
ax.yaxis.set_tick_params(width=5, length=10)
ax.tick_params(axis='x', labelsize=22)
ax.tick_params(axis='y', labelsize=22)
plt.xticks([0, 2, 4, 6, 8, 10, 12])
plt.legend(loc="best", fontsize=18)
plt.ylabel('log10(Protein count)', fontsize=25)
plt.xlabel('log2(TPM)', fontsize=25)
plt.savefig('figures/lin_models/corr_IL10.pdf', bbox_inches='tight', transparent=True)
plt.close()

/home/qmarti/miniconda3/envs/env_data/lib/python3.10/site-packages/sklearn/utils/validation.py:1406: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/home/qmarti/miniconda3/envs/env_data/lib/python3.10/site-packages/sklearn/utils/validation.py:1406: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/home/qmarti/miniconda3/envs/env_data/lib/python3.10/site-packages/sklearn/utils/validation.py:1406: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/home/qmarti/miniconda3/envs/env_data/lib/python3.10/site-packages/sklearn/utils/validation.py:1406: DataConversionWarning: A colu

I10R1_HUMAN: MAE= 0.13224538760679858
I10R1_HUMAN: PCC= PearsonRResult(statistic=0.7977791238683013, pvalue=0.00998409487383745)
I10R2_HUMAN: MAE= 0.22679984879430207
I10R2_HUMAN: PCC= PearsonRResult(statistic=0.7441355906275132, pvalue=4.683251364655821e-05)
STAT1_HUMAN: MAE= 0.21075743744569364
STAT1_HUMAN: PCC= PearsonRResult(statistic=0.7195034395261929, pvalue=3.43398110535602e-05)
STAT3_HUMAN: MAE= 0.22363285198633104
STAT3_HUMAN: PCC= PearsonRResult(statistic=0.515482199451881, pvalue=0.007035075533588906)


In [4]:
# For data points without proteomic data, use the linear models to estimate it
for gene in gene_list:
    df.loc[(df["Gene"]==gene)&(df["Prot. count dataset num."].isna()),"Log10 Prot. count"] = df_lin.loc[df_lin["Gene"]==gene,"Intercept"].values[0]+df.loc[(df["Gene"]==gene)&(df["Prot. count dataset num."].isna()),"Log2 TPM"]*df_lin.loc[df_lin["Gene"]==gene,"Slope"].values[0]
    df.loc[(df["Gene"]==gene)&(df["Prot. count dataset num."].isna()),"Prot. count dataset num."] = 0
df.to_csv("data/expression/whole_dataset_IL10.tsv.gz", sep="\t", index=False, compression="gzip") 

In [5]:
print("IC for M (accessory molecule for STAT1 binding): "+str(10**df.loc[(df["Prot. count dataset num."]!=0)&(df["Gene"].isin(["STAT1_HUMAN","STAT3_HUMAN"])),"Log10 Prot. count"].mean()))

IC for M (accessory molecule for STAT1 binding): 226711.45578971403


### Get error of linear models in test dataset (tissues)

In [6]:
# Get linear models and datasets
df_immune = pd.read_csv('data/expression/immune_dataset.csv').dropna()
df_tissues = pd.read_csv("data/expression/tissues_dataset.csv").dropna()
df_lin = pd.read_csv('data/lin_models/linear_models_IL10.csv',index_col=0)

In [7]:
# Get transcriptomic and proteomic data from 4/5 genes in the model (No more available data on IL-10RA)
df_test = df_tissues.loc[(df_tissues["Gene"]=="I10R2_HUMAN")]
df_test = pd.concat([df_test, df_tissues.loc[(df_tissues["Gene"]=="STAT1_HUMAN")]], ignore_index=True)
df_test = pd.concat([df_test, df_tissues.loc[(df_tissues["Gene"]=="STAT3_HUMAN")]], ignore_index=True)
# Remove duplicate entries
df_test = df_test.loc[(df_test["Cell"]+df_test["Gene"]).drop_duplicates(keep="last").index]
# Get estimation of protein count
df_test["Log10 Prot. count (estim.)"] = [df_lin.loc[df_lin["Gene"]==df_test.loc[index,"Gene"],"Intercept"].values[0]+df_test.loc[index,"Log2 TPM"]*df_lin.loc[df_lin["Gene"]==df_test.loc[index,"Gene"],"Slope"].values[0] for index in df_test.index]
# Get mean absolute error
df_test["MAE"] = abs(df_test["Log10 Prot. count"]-df_test["Log10 Prot. count (estim.)"])

In [8]:
print("MAE: "+str(df_test["MAE"].mean()))
print("PCC TPMxProt:"+str(pearsonr(df_test["Log2 TPM"],df_test["Log10 Prot. count"]).statistic))
print("PCC ModelxProt:"+str(pearsonr(df_test["Log10 Prot. count (estim.)"],df_test["Log10 Prot. count"]).statistic))

MAE: 0.40340045440054745
PCC TPMxProt:0.4508705410148255
PCC ModelxProt:0.9413372000745961


In [9]:
fig, ax = plt.subplots(1,1,figsize=(9, 8), dpi=600)
i = 1
for gene in df_test["Gene"].drop_duplicates():
    plt.scatter(df_test.loc[df_test["Gene"]==gene,"Log10 Prot. count"].values, df_test.loc[df_test["Gene"]==gene,"Log10 Prot. count (estim.)"].values, s=60, alpha = 1, label=gene.split("_HUMAN")[0], color=color_sim[i])
    i += 1
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.xaxis.set_tick_params(width=5, length=10)
ax.yaxis.set_tick_params(width=5, length=10)
ax.tick_params(axis='x', labelsize=22)
ax.tick_params(axis='y', labelsize=22)
plt.legend(loc="best", fontsize=18)
plt.xticks([3, 4, 5, 6])
plt.yticks([3, 4, 5, 6])
plt.xlabel('Log10 Prot. count', fontsize=25)
plt.ylabel('Log10 Prot. count (predict)', fontsize=25)
plt.savefig('figures/lin_models/corr_IL10_test.pdf', bbox_inches='tight', transparent=True)
plt.close()

In [10]:
# Get Pearson correlation per gene:
print(pearsonr(df_test.loc[df_test["Gene"]=="I10R2_HUMAN"]["Log10 Prot. count"],df_test.loc[df_test["Gene"]=="I10R2_HUMAN"]["Log10 Prot. count (estim.)"]))
print(pearsonr(df_test.loc[df_test["Gene"]=="STAT1_HUMAN"]["Log10 Prot. count"],df_test.loc[df_test["Gene"]=="STAT1_HUMAN"]["Log10 Prot. count (estim.)"]))
print(pearsonr(df_test.loc[df_test["Gene"]=="STAT3_HUMAN"]["Log10 Prot. count"],df_test.loc[df_test["Gene"]=="STAT3_HUMAN"]["Log10 Prot. count (estim.)"]))

PearsonRResult(statistic=0.08582835911024742, pvalue=0.6900683782501158)
PearsonRResult(statistic=0.11701293946400865, pvalue=0.5455122068841524)
PearsonRResult(statistic=0.20329079175244621, pvalue=0.2901950895743608)


In [11]:
# Get Pearson correlation per gene:
print(pearsonr(df_test.loc[df_test["Gene"]=="I10R2_HUMAN"]["Log10 Prot. count"],df_test.loc[df_test["Gene"]=="I10R2_HUMAN"]["Log10 Prot. count (estim.)"]))
print(pearsonr(df_test.loc[df_test["Gene"]=="STAT1_HUMAN"]["Log10 Prot. count"],df_test.loc[df_test["Gene"]=="STAT1_HUMAN"]["Log10 Prot. count (estim.)"]))
print(pearsonr(df_test.loc[df_test["Gene"]=="STAT3_HUMAN"]["Log10 Prot. count"],df_test.loc[df_test["Gene"]=="STAT3_HUMAN"]["Log10 Prot. count (estim.)"]))

PearsonRResult(statistic=0.08582835911024742, pvalue=0.6900683782501158)
PearsonRResult(statistic=0.11701293946400865, pvalue=0.5455122068841524)
PearsonRResult(statistic=0.20329079175244621, pvalue=0.2901950895743608)


In [12]:
# Get Pearson correlation per gene:
print((df_test.loc[df_test["Gene"]=="I10R2_HUMAN"]["Log10 Prot. count"]-df_test.loc[df_test["Gene"]=="I10R2_HUMAN"]["Log10 Prot. count (estim.)"]).abs().mean())
print((df_test.loc[df_test["Gene"]=="STAT1_HUMAN"]["Log10 Prot. count"]-df_test.loc[df_test["Gene"]=="STAT1_HUMAN"]["Log10 Prot. count (estim.)"]).abs().mean())
print((df_test.loc[df_test["Gene"]=="STAT3_HUMAN"]["Log10 Prot. count"]-df_test.loc[df_test["Gene"]=="STAT3_HUMAN"]["Log10 Prot. count (estim.)"]).abs().mean())

0.3612861137769367
0.45593906505130194
0.38571509116243646


### Compare results between train (immune cell types and cancer cell lines) and test (human tissues)

In [13]:
# Get training data, compute response and get MAE to compare with test data
df = pd.read_csv("data/expression/whole_dataset_IL10.tsv.gz", sep="\t", compression="gzip")
df_train = df.loc[df["Prot. count dataset num."]!=0]
df_train["Log10 Prot. count (estim.)"] = [df_lin.loc[df_lin["Gene"]==df_train.loc[index,"Gene"],"Intercept"].values[0]+df_train.loc[index,"Log2 TPM"]*df_lin.loc[df_lin["Gene"]==df_train.loc[index,"Gene"],"Slope"].values[0] for index in df_train.index]
df_train["MAE"] = abs(df_train["Log10 Prot. count"]-df_train["Log10 Prot. count (estim.)"])

/tmp/ipykernel_18762/2509673992.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_train["Log10 Prot. count (estim.)"] = [df_lin.loc[df_lin["Gene"]==df_train.loc[index,"Gene"],"Intercept"].values[0]+df_train.loc[index,"Log2 TPM"]*df_lin.loc[df_lin["Gene"]==df_train.loc[index,"Gene"],"Slope"].values[0] for index in df_train.index]
/tmp/ipykernel_18762/2509673992.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_train["MAE"] = abs(df_train["Log10 Prot. count"]-df_train["Log10 Prot. count (estim.)"])


In [14]:
# Is there a significant difference in the error between train and test sets
f_statistic, p_value = stats.ttest_ind(df_test["MAE"].dropna().values, df_train["MAE"].dropna().values, equal_var=False, alternative="greater")
fig, ax = plt.subplots(1, 1, figsize=(5, 7), dpi=400)

ax.boxplot(
    [
        df_train["MAE"].dropna().to_list(),
        df_test["MAE"].dropna().to_list()
    ],
    widths=0.6,
    patch_artist=True,
    boxprops=dict(facecolor="grey", edgecolor="black", linewidth=1.5),
    medianprops=dict(color="black", linewidth=1.5),
    whiskerprops=dict(color="black", linewidth=1.5),
    capprops=dict(color="black", linewidth=1.5)
)

# Spine formatting
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Tick formatting
ax.xaxis.set_tick_params(width=4, length=8)
ax.yaxis.set_tick_params(width=4, length=8)
ax.tick_params(axis="x", labelsize=17)
ax.tick_params(axis="y", labelsize=17)

# Labels
ax.set_ylabel("$MAE$", fontsize=20)
ax.set_xticklabels(["Training set", "Test set"], fontsize=17)

plt.tight_layout()
plt.savefig(
    "/users/lserrano/qmarti/PhD_code/IL10/figures/lin_models/Plot_box_train_test.pdf",
    transparent=True,
    bbox_inches="tight"
)
plt.close(fig)
print("Train vs Test linear models p value: "+str(p_value))
print("N: "+str([len(df_train["MAE"].dropna().values),len(df_test["MAE"].dropna().values)]))

Train vs Test linear models p value: 3.0467523608593956e-06
N: [84, 82]


### Get correaltion of linear models in test dataset (cancer cell lines, AUC)

In [15]:
# Get protein abundance datasets and combine them
df_test1 = pd.read_csv('data/expression/CCLE_prot_abundance_dataset.tsv.gz', sep='\t')
df_test1 = df_test1.loc[df_test1["Gene"].isin(["I10R2_HUMAN","STAT1_HUMAN","STAT3_HUMAN"])]
df_test1["Dataset name"] = "CCLE_Gygi"
df_test = pd.read_csv('data/expression/cells_CCLE_Olink.csv')
df_test = df_test.loc[df_test["Log2 TPM"]>0.25]
for gene in ["I10R1_HUMAN","I10R2_HUMAN","STAT1_HUMAN"]:
    df_test.loc[df_test["Gene"]==gene,"Abundance"] = df_test.loc[df_test["Gene"]==gene,"Abundance"] - df_test.loc[df_test["Gene"]==gene,"Abundance"].mean()
df_test["Dataset name"] = "CCLE_Olink"
df_test = pd.concat([df_test1,df_test]).reset_index(drop=True)

# Get protein estimations
df_test["Log10 Prot. count (estim.)"] = [df_lin.loc[df_lin["Gene"]==df_test.loc[index,"Gene"],"Intercept"].values[0]+df_test.loc[index,"Log2 TPM"]*df_lin.loc[df_lin["Gene"]==df_test.loc[index,"Gene"],"Slope"].values[0] for index in df_test.index]

In [16]:
color_sim = [(68/255, 138/255, 255/255),(0, 150/255, 136/255),(139/255, 195/255, 74/255),(255/255, 193/255, 7/255),(255/255, 152/255, 0),(244/255, 67/255, 54/255),(173/255, 20/255, 87/255)]
fig, ax = plt.subplots(1, 1, figsize=(9, 8), dpi=600)
for i, gene in enumerate(["I10R1_HUMAN","I10R2_HUMAN","STAT1_HUMAN","STAT3_HUMAN"]):
    df_gene = df_test.loc[df_test["Gene"] == gene]
    plt.scatter(df_gene["Abundance"], df_gene["Log10 Prot. count (estim.)"],
                s=60, color=color_sim[i], alpha=1, label=gene.split("_HUMAN")[0])
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.xaxis.set_tick_params(width=5, length=10)
ax.yaxis.set_tick_params(width=5, length=10)
ax.tick_params(axis='x', labelsize=22)
ax.tick_params(axis='y', labelsize=22)
plt.legend(loc="best", fontsize=18)
plt.ylabel('log10(Protein count) (estim.)', fontsize=25)
plt.xlabel('Relative Abundance', fontsize=25)
plt.savefig('figures/lin_models/corr_IL10_test_CCLE.pdf', bbox_inches='tight', transparent=True)
plt.close()

In [17]:
# Get Pearson correlation per gene:
print(pearsonr(df_test.loc[df_test["Gene"]=="I10R1_HUMAN"]["Abundance"],df_test.loc[df_test["Gene"]=="I10R1_HUMAN"]["Log10 Prot. count (estim.)"]))
print(pearsonr(df_test.loc[df_test["Gene"]=="I10R2_HUMAN"]["Abundance"],df_test.loc[df_test["Gene"]=="I10R2_HUMAN"]["Log10 Prot. count (estim.)"]))
print(pearsonr(df_test.loc[df_test["Gene"]=="STAT1_HUMAN"]["Abundance"],df_test.loc[df_test["Gene"]=="STAT1_HUMAN"]["Log10 Prot. count (estim.)"]))
print(pearsonr(df_test.loc[df_test["Gene"]=="STAT3_HUMAN"]["Abundance"],df_test.loc[df_test["Gene"]=="STAT3_HUMAN"]["Log10 Prot. count (estim.)"]))

PearsonRResult(statistic=0.7143649754159811, pvalue=2.085227408068696e-06)
PearsonRResult(statistic=0.3142612407533438, pvalue=9.204051530914513e-10)
PearsonRResult(statistic=0.4909347016158353, pvalue=4.0332550732440265e-30)
PearsonRResult(statistic=0.6152160374653185, pvalue=1.7282076345021554e-39)


### Plot IL-10RA and IL-10RB expression in all assayed cell types/cell lines

In [18]:
df = pd.read_csv("data/expression/whole_dataset_IL10.tsv.gz", sep='\t', compression='gzip')
cells_assayed = ['NK','ACH-000786','ACH-000995','ACH-000146','MO.Mean','B.Mean','T4.Mean','T8.Mean','BLaER1','Haftl']
x = df.loc[(df["Gene"] == "I10R1_HUMAN") & (df["Cell"].isin(cells_assayed)),"Log2 TPM"]
y = df.loc[(df["Gene"] == "I10R2_HUMAN") & (df["Cell"].isin(cells_assayed)),"Log2 TPM"]
cells = ['THP-1','Daudi','Jurkat','NK','BLaER1','Haftl','B cells','Monocytes','CD4+ T cells','CD8+ T cells']

fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
for xi, yi, cell in zip(x, y, cells):
    plt.text(
        xi+0.15, yi, cell,
        fontsize=15,
        ha="left",
        va="center"
    )
x_train = df.loc[(df["Gene"] == "I10R1_HUMAN") & (df["Cell"].isin(["T8.Mean","T4.Mean","MO.Mean","ACH-000146","ACH-000786"])),"Log2 TPM"]
y_train = df.loc[(df["Gene"] == "I10R2_HUMAN") & (df["Cell"].isin(["T8.Mean","T4.Mean","MO.Mean","ACH-000146","ACH-000786"])),"Log2 TPM"]
x_test = df.loc[(df["Gene"] == "I10R1_HUMAN") & (df["Cell"].isin(['BLaER1','Haftl','B.Mean','ACH-000995','NK'])),"Log2 TPM"]
y_test = df.loc[(df["Gene"] == "I10R2_HUMAN") & (df["Cell"].isin(['BLaER1','Haftl','B.Mean','ACH-000995','NK'])),"Log2 TPM"]
plt.scatter(x_train,y_train,color="darkred",s=200,label="Training pSTAT data")
plt.scatter(x_test,y_test,color="darkblue",s=200,label="Test pSTAT data")
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_xticks([])
ax.set_yticks([])
plt.xlabel("IL-10R1 expression", fontsize=25)
plt.ylabel("IL-10R2 expression", fontsize=25)
plt.legend(loc="lower right", fontsize=15)
plt.savefig('figures/lin_models/expression_RA_RB_all_cells.pdf', transparent=True, bbox_inches="tight")
plt.close()